In [1]:
# Cell 1: Verify GPU and install dependencies
!nvidia-smi

!pip install -q -U git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen-vl-utils pillow

Sun May 10 01:17:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Clone dataset, copy images, and build stable tasks.json

!rm -rf /content/source_repo /content/images
!git clone --depth 1 https://github.com/vishwas182002/Financial-Document-VQA.git /content/source_repo

!cp /content/source_repo/data/financial_test/annotations/combined_financial_vqa_dataset_fixed_types.json /content/
!cp -r /content/source_repo/data/financial_test/images/samples /content/images

from pathlib import Path
import json

RAW_PATH = Path("/content/combined_financial_vqa_dataset_fixed_types.json")
IMAGES_ROOT = Path("/content/images")
TASKS_PATH = Path("/content/tasks.json")

with RAW_PATH.open() as f:
    raw = json.load(f)

tasks = []
counters = {}
missing_images = []

for entry in raw:
    ticker = entry["ticker"]
    company = entry["company"]
    source_section = entry.get("section", "unknown")
    image_type = entry.get("image_type", "unknown")
    image_path = IMAGES_ROOT / ticker / entry["image_path"]

    if not image_path.exists():
        missing_images.append(str(image_path))

    for q in entry["questions"]:
        counters[ticker] = counters.get(ticker, 0) + 1
        task_id = f"{ticker}_{counters[ticker]:03d}"

        tasks.append({
            "task_id": task_id,
            "track": "track_a",
            "ticker": ticker,
            "company": company,
            "image_path": str(image_path),
            "question": q["question"],
            "gold_answer": q["answer"],
            "category": q["type"],
            "source_section": source_section,
            "image_type": image_type,
        })

with TASKS_PATH.open("w") as f:
    json.dump(tasks, f, indent=2)

print(f"Raw entries: {len(raw)}")
print(f"Tasks: {len(tasks)}")
print(f"Unique task_ids: {len({t['task_id'] for t in tasks})}")
print(f"Unique tickers: {len(counters)}")
print(f"PNG images: {len(list(IMAGES_ROOT.rglob('*.png')))}")
print(f"Missing images: {len(missing_images)}")
print(f"Sample task_ids: {tasks[0]['task_id']}, {tasks[-1]['task_id']}")

assert len(raw) == 79
assert len(tasks) == 397
assert len({t["task_id"] for t in tasks}) == 397
assert len(counters) == 10
assert len(list(IMAGES_ROOT.rglob("*.png"))) == 79
assert not missing_images

print("Data setup OK.")


Cloning into '/content/source_repo'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 138 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 19.44 MiB | 20.25 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Raw entries: 79
Tasks: 397
Unique task_ids: 397
Unique tickers: 10
PNG images: 79
Missing images: 0
Sample task_ids: AAPL_001, JPM_035
Data setup OK.


In [4]:
# Repair Cell: fix Pillow / torchvision import mismatch

!pip install -q --force-reinstall "pillow==11.3.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 134.5 MB/s eta 0:00:00


In [5]:
# Import sanity check after runtime restart

import torch
import PIL
import torchvision
import transformers
import bitsandbytes as bnb

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

print("Pillow:", PIL.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Imports OK")


Pillow: 11.3.0
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
transformers: 5.8.0.dev0
cuda available: True
gpu: Tesla T4
Imports OK


In [6]:
# Cell 3: Load Qwen2.5-VL-7B-Instruct in 4-bit

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

min_pixels = 256 * 28 * 28
max_pixels = 1024 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
)

print("Model loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("GPU memory allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
print("GPU memory reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-VL-7B-Instruct
Device: cuda:0
GPU memory allocated GB: 5.51
GPU memory reserved GB: 7.64


In [7]:
# Cell 4: Single-task smoke test

import json
import time
import torch
from pathlib import Path
from qwen_vl_utils import process_vision_info

with open("/content/tasks.json") as f:
    tasks = json.load(f)

def predict_qwen(task, max_new_tokens=64):
    prompt = (
        "Answer the question using only the document image. "
        "Return only the final answer, with no explanation.\n\n"
        f"Question: {task['question']}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": task["image_path"]},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    start = time.time()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    latency_ms = round((time.time() - start) * 1000)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return output, latency_ms

task = tasks[0]
prediction, latency_ms = predict_qwen(task)

print("Task ID:", task["task_id"])
print("Ticker:", task["ticker"])
print("Category:", task["category"])
print("Question:", task["question"])
print("Gold:", task["gold_answer"])
print("Prediction:", prediction)
print("Latency ms:", latency_ms)
print("GPU memory allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


Task ID: AAPL_001
Ticker: AAPL
Category: extractive
Question: What was the total net sales for the three months ended December 27, 2025?
Gold: 143,756
Prediction: 143,756
Latency ms: 5378
GPU memory allocated GB: 5.52


In [8]:
# Cell 5: Run Qwen2.5-VL on all 397 tasks with checkpointing

import json
import time
import gc
import torch
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_PATH = Path("/content/qwen25vl_7b_results.json")
CHECKPOINT_PATH = Path("/content/qwen25vl_7b_results_checkpoint.json")

MODEL_NAME = "Qwen2.5-VL-7B-Instruct"

# Resume if checkpoint exists
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open() as f:
        results = json.load(f)
    completed_ids = {r["task_id"] for r in results}
    print(f"Resuming from checkpoint: {len(results)} completed")
else:
    results = []
    completed_ids = set()
    print("Starting fresh")

def save_checkpoint(path=CHECKPOINT_PATH):
    with path.open("w") as f:
        json.dump(results, f, indent=2)

for task in tqdm(tasks, total=len(tasks)):
    if task["task_id"] in completed_ids:
        continue

    try:
        prediction, latency_ms = predict_qwen(task, max_new_tokens=64)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM on {task['task_id']}; clearing cache and retrying with fewer tokens...")
        torch.cuda.empty_cache()
        gc.collect()
        prediction, latency_ms = predict_qwen(task, max_new_tokens=32)
    except Exception as e:
        print(f"ERROR on {task['task_id']}: {repr(e)}")
        prediction = ""
        latency_ms = None

    results.append({
        "task_id": task["task_id"],
        "model": MODEL_NAME,
        "prediction": prediction,
        "gold_answer": task["gold_answer"],
        "category": task["category"],
        "latency_ms": latency_ms,
        "evidence_available": False,
        "evidence": None,
        "raw_output": prediction,
    })

    completed_ids.add(task["task_id"])

    if len(results) % 25 == 0:
        save_checkpoint()
        print(f"Saved checkpoint: {len(results)}/{len(tasks)}")

save_checkpoint(OUTPUT_PATH)

print(f"Done. Wrote {len(results)} results to {OUTPUT_PATH}")
print("Average latency ms:", round(sum(r["latency_ms"] for r in results if r["latency_ms"] is not None) / sum(1 for r in results if r["latency_ms"] is not None), 1))
print("Output:", OUTPUT_PATH)


Starting fresh


  0%|          | 0/397 [00:00<?, ?it/s]

Saved checkpoint: 25/397
Saved checkpoint: 50/397
Saved checkpoint: 75/397
Saved checkpoint: 100/397
Saved checkpoint: 125/397
Saved checkpoint: 150/397
Saved checkpoint: 175/397
Saved checkpoint: 200/397
Saved checkpoint: 225/397
Saved checkpoint: 250/397
Saved checkpoint: 275/397
Saved checkpoint: 300/397
Saved checkpoint: 325/397
Saved checkpoint: 350/397
Saved checkpoint: 375/397
Done. Wrote 397 results to /content/qwen25vl_7b_results.json
Average latency ms: 3075.2
Output: /content/qwen25vl_7b_results.json


In [9]:
# Cell 6: Download Qwen results

from google.colab import files

files.download("/content/qwen25vl_7b_results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Cell 7: Reliability Mini-Probe — 25 questions x 3 paraphrases

import json
import random
import re
import time
import torch
from pathlib import Path
from tqdm.auto import tqdm

assert "model" in globals(), "Model is not loaded. Rerun the model-loading cell."
assert "processor" in globals(), "Processor is not loaded. Rerun the model-loading cell."
assert "predict_qwen" in globals(), "predict_qwen is missing. Rerun the smoke-test cell."

random.seed(42)

with open("/content/tasks.json") as f:
    tasks = json.load(f)


# -----------------------------
# Answer normalization
# -----------------------------

def normalize_text(text):
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"^(answer:|the answer is|it is)\s*", "", text)
    text = text.strip().strip(".")
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_financial_number(text):
    if not text:
        return None

    s = str(text).strip()
    s = re.sub(r"^(answer:|the answer is|it is)\s*", "", s, flags=re.IGNORECASE)
    s = s.strip().strip(".")

    paren_match = re.match(r"^\((.+)\)$", s)
    if paren_match:
        s = "-" + paren_match.group(1)

    s = re.sub(r"[\$€£]", "", s)

    is_percent = s.endswith("%")
    if is_percent:
        s = s[:-1].strip()

    s = s.replace(",", "").strip()

    suffix_multipliers = {
        "k": 1_000,
        "m": 1_000_000,
        "b": 1_000_000_000,
        "t": 1_000_000_000_000,
    }

    suffix_match = re.match(r"^(-?[\d.]+)\s*([kmbt])$", s, re.IGNORECASE)
    if suffix_match:
        num_str = suffix_match.group(1)
        suffix = suffix_match.group(2).lower()
        try:
            s = str(float(num_str) * suffix_multipliers[suffix])
        except ValueError:
            return None

    try:
        val = float(s)
        if val == int(val) and not is_percent:
            result = str(int(val))
        else:
            result = str(val)
        return f"{result}%" if is_percent else result
    except ValueError:
        return None


def canonical_answer(answer):
    num = normalize_financial_number(answer)
    if num is not None:
        return f"NUM::{num}"
    return f"TXT::{normalize_text(answer)}"


def answers_match(a, b):
    return canonical_answer(a) == canonical_answer(b)


# -----------------------------
# Stratified task selection
# -----------------------------

by_category = {}
for task in tasks:
    by_category.setdefault(task["category"], []).append(task)

sample_counts = {
    "extractive": 8,
    "layout_understanding": 6,
    "numerical_reasoning": 6,
    "chart_interpretation": 5,
}

probe_tasks = []
for category, count in sample_counts.items():
    pool = by_category[category]
    probe_tasks.extend(random.sample(pool, min(count, len(pool))))

print(f"Selected {len(probe_tasks)} probe tasks:")
for category in sample_counts:
    actual = sum(1 for t in probe_tasks if t["category"] == category)
    print(f"  {category}: {actual}")


# -----------------------------
# Generate paraphrases
# -----------------------------

def parse_numbered_lines(raw, n=3):
    paraphrases = []
    for line in raw.splitlines():
        line = line.strip()
        match = re.match(r"^\s*\d+[\.\)\-:]\s*(.+)$", line)
        if match:
            cleaned = match.group(1).strip().strip('"')
            if cleaned:
                paraphrases.append(cleaned)
    return paraphrases[:n]


def fallback_paraphrases(question):
    return [
        f"Using only the document image, {question}",
        f"From the shown financial document, {question}",
        f"According to the filing screenshot, {question}",
    ]


def generate_paraphrases(question, n=3):
    prompt = (
        f"Rewrite the following financial document question {n} different ways. "
        f"Keep the exact same meaning, numbers, dates, and entities. "
        f"Do not answer the question. "
        f"Return only a numbered list from 1 to {n}.\n\n"
        f"Original question: {question}"
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=False,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    raw = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    paraphrases = parse_numbered_lines(raw, n=n)

    if len(paraphrases) < n:
        paraphrases = fallback_paraphrases(question)

    return paraphrases[:n]


print("\nGenerating paraphrases...")
probe_data = []

for task in tqdm(probe_tasks, desc="Paraphrasing"):
    paraphrases = generate_paraphrases(task["question"], n=3)
    probe_data.append({
        "task": task,
        "original_question": task["question"],
        "paraphrases": paraphrases,
    })

print(f"Generated paraphrases for {len(probe_data)} tasks")


# -----------------------------
# Run original + paraphrased inference
# -----------------------------

print("\nRunning reliability inference...")
results = []

for item in tqdm(probe_data, desc="Probe inference"):
    task = item["task"]

    variants = [{"variant": "original", "question": item["original_question"]}]
    for i, paraphrase in enumerate(item["paraphrases"], start=1):
        variants.append({"variant": f"paraphrase_{i}", "question": paraphrase})

    variant_results = []

    for variant in variants:
        variant_task = {**task, "question": variant["question"]}
        pred, latency_ms = predict_qwen(variant_task)

        variant_results.append({
            "variant": variant["variant"],
            "question": variant["question"],
            "prediction": pred,
            "canonical_prediction": canonical_answer(pred),
            "correct": answers_match(pred, task["gold_answer"]),
            "latency_ms": latency_ms,
        })

    canonical_answers = [v["canonical_prediction"] for v in variant_results]
    correctness_values = [v["correct"] for v in variant_results]

    consistent_answer = len(set(canonical_answers)) == 1
    correctness_stable = len(set(correctness_values)) == 1

    results.append({
        "task_id": task["task_id"],
        "ticker": task["ticker"],
        "category": task["category"],
        "gold_answer": task["gold_answer"],
        "canonical_gold": canonical_answer(task["gold_answer"]),
        "original_question": item["original_question"],
        "paraphrases": item["paraphrases"],
        "variants": variant_results,
        "consistent_answer": consistent_answer,
        "correctness_stable": correctness_stable,
        "original_correct": variant_results[0]["correct"],
        "all_correct": all(correctness_values),
        "any_correct": any(correctness_values),
    })


# -----------------------------
# Report summary
# -----------------------------

total = len(results)
consistent_count = sum(r["consistent_answer"] for r in results)
stable_count = sum(r["correctness_stable"] for r in results)
original_correct = sum(r["original_correct"] for r in results)
all_correct = sum(r["all_correct"] for r in results)
any_correct = sum(r["any_correct"] for r in results)

print(f"\n{'=' * 70}")
print("RELIABILITY MINI-PROBE")
print(f"{'=' * 70}")
print(f"Tasks: {total}")
print(f"Variants per task: 4")
print(f"Total inferences: {total * 4}")
print(f"Answer consistency: {consistent_count}/{total} ({100 * consistent_count / total:.1f}%)")
print(f"Correctness stable: {stable_count}/{total} ({100 * stable_count / total:.1f}%)")
print(f"Original-question accuracy: {original_correct}/{total} ({100 * original_correct / total:.1f}%)")
print(f"All-variants correct: {all_correct}/{total} ({100 * all_correct / total:.1f}%)")
print(f"Any-variant correct: {any_correct}/{total} ({100 * any_correct / total:.1f}%)")

print("\nPer-category consistency:")
for category in sample_counts:
    cat_results = [r for r in results if r["category"] == category]
    cat_consistent = sum(r["consistent_answer"] for r in cat_results)
    cat_all_correct = sum(r["all_correct"] for r in cat_results)
    print(
        f"  {category}: "
        f"consistent {cat_consistent}/{len(cat_results)}, "
        f"all-correct {cat_all_correct}/{len(cat_results)}"
    )

inconsistent = [r for r in results if not r["consistent_answer"]]
if inconsistent:
    print(f"\nInconsistent examples ({len(inconsistent)} total, showing first 5):")
    for r in inconsistent[:5]:
        print(f"\nTask: {r['task_id']} ({r['category']})")
        print(f"Gold: {r['gold_answer']}")
        print(f"Original Q: {r['original_question']}")
        for v in r["variants"]:
            print(f"  {v['variant']}: {v['prediction']} | correct={v['correct']}")

OUT_PATH = "/content/reliability_probe_results.json"
with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {OUT_PATH}")


Selected 25 probe tasks:
  extractive: 8
  layout_understanding: 6
  numerical_reasoning: 6
  chart_interpretation: 5

Generating paraphrases...


Paraphrasing:   0%|          | 0/25 [00:00<?, ?it/s]

Generated paraphrases for 25 tasks

Running reliability inference...


Probe inference:   0%|          | 0/25 [00:00<?, ?it/s]


RELIABILITY MINI-PROBE
Tasks: 25
Variants per task: 4
Total inferences: 100
Answer consistency: 11/25 (44.0%)
Correctness stable: 20/25 (80.0%)
Original-question accuracy: 13/25 (52.0%)
All-variants correct: 9/25 (36.0%)
Any-variant correct: 14/25 (56.0%)

Per-category consistency:
  extractive: consistent 4/8, all-correct 3/8
  layout_understanding: consistent 5/6, all-correct 5/6
  numerical_reasoning: consistent 0/6, all-correct 0/6
  chart_interpretation: consistent 2/5, all-correct 1/5

Inconsistent examples (14 total, showing first 5):

Task: AMZN_022 (extractive)
Gold: 37,855
Original Q: What were the net sales in the United Kingdom in 2024?
  original: 37,586 | correct=False
  paraphrase_1: 37,586 | correct=False
  paraphrase_2: $81,984 | correct=False
  paraphrase_3: The image does not contain information about the gross income recorded in the UK for the period from January 1, 2024, to December 31, 2024. The provided data is about net additions and depreciation and amortizati

In [11]:
from google.colab import files
files.download("/content/reliability_probe_results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>